<a href="https://colab.research.google.com/github/smargetic/Natural_Language_Processing/blob/main/Machine_Translation/Translation_Model_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# English → Italian Translation: Seq2Seq with Attention (PyTorch)

The recurrent cells, attention and beam search are written by hand so every step is visible.

**This notebook defines the model; it doesn't train anything by itself.** Running it top to bottom takes a few seconds and creates all the functions and classes. Training, comparing configurations and keeping the best model happen in **`Experiments.ipynb`**, which loads this notebook with `%run Translation_Model_Pytorch.ipynb`.

**Sections**
1. Setup and settings
2. Preparing the data: from sentence pairs to batches (load, clean, split, segment, vocabulary)
3. Building blocks: embedding, RNN cell, LSTM cell
4. Encoder
5. Attention
6. Decoder
7. Seq2Seq (encoder + decoder together)
8. Beam search (turning the model's scores into a sentence)
9. Training loop
10. Evaluation (loss + BLEU)
11. Putting it together: `run_experiment(config)`
12. Transformer: an alternative to the RNN / LSTM

**Shape names used in the comments**

| letter | meaning |
|---|---|
| `B` | batch size (number of sentences) |
| `S` | source length (English words, including `<sos>`/`<eos>`/padding) |
| `T` | target length (Italian words, including `<sos>`/`<eos>`/padding) |
| `E` | embedding size |
| `H` | hidden size |
| `V` | vocabulary size |

## 1. Setup and settings

In [1]:
import os
import sys
import csv
import re
import math
import time
import random
import zipfile
from collections import Counter, defaultdict, namedtuple

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

#show all columns/text in pandas tables
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

#disable scientific notation when printing tensors
torch.set_printoptions(sci_mode=False)

#running in Google Colab? -> mount Google Drive and move into the project folder
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  os.chdir('/content/drive/My Drive/Coding Projects/Machine Translation')   #<- change this if the folder moves

#use the GPU if there is one (in Colab: Runtime -> Change runtime type -> GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

#same random numbers on every run -> results you can compare between runs
SEED = 42
random.seed(SEED)
_ = torch.manual_seed(SEED)

Using device: cpu


The **default settings**, all in one place. `Experiments.ipynb` starts from these and changes a few at a time, so each experiment is described only by what it changes.

The defaults are small, so one epoch takes about 5 minutes on a laptop CPU. For a better model, raise `sample_frac`, `n_epochs` and the hidden sizes, and use a GPU.

In [2]:
CONFIG = {
  ### data ###
  'data_file': 'Sentence pairs in English-Italian - 2024-07-26.tsv.zip',   #the .zip or the unzipped .tsv both work
  'sample_frac': 0.05,        #fraction of the ~628k sentence pairs to use (0.05 = 5% = ~31k pairs)
  'max_sentence_words': 15,   #skip longer sentences (counted in words + punctuation, without <sos>/<eos>)
  'min_word_count': 1,        #tokens seen fewer times than this in the training data become <unk> (1 = keep every token)

  ### how each language's words are split into tokens - see "Segmentation methods" in section 2 ###
  #  'words'       whole words                                  andremo -> andremo
  #  'bpe'         byte-pair encoding subwords                  andremo -> andr@@ emo   (split learned from frequency)
  #  'stem+bpe'    stemmer cuts off the ending, then BPE        andremo -> andr@@ emo   (split from grammar rules; needs nltk)
  #  'lemma+tags'  dictionary form + grammar tag                parliamo -> parlare #VERB:...Person=1   (needs spaCy + language model)
  'segmentation': {'english': 'words', 'italian': 'bpe'},
  'bpe_merges': 4000,         #merges BPE learns per language ('bpe' and 'stem+bpe') - more merges = longer pieces, bigger vocabulary
  'val_frac': 0.1,           #10% of English sentences for validation (checking progress during training)
  'test_frac': 0.1,           #10% for the final test (only used at the very end)
  'batch_size': 64,

  ### model ###
  'node_type': 'lstm',        #'rnn', 'lstm' or 'transformer' (section 12)
  'embed_dim': 128,
  'enc_hidden_dim': 256,
  'dec_hidden_dim': 256,
  'enc_n_layers': 1,
  'bidirectional': False,     #RNN / LSTM only: the encoder also reads each sentence right to left (section 4)
  'dec_n_layers': 1,
  'attention': True,
  'sim_method': 'scaled_dot', #'dot', 'scaled_dot', or 'bahdanau'
  'attn_dim': 128,            #size keys and queries are projected to before comparing them

  ### Transformer only (node_type = 'transformer'); layers come from enc_n_layers / dec_n_layers ###
  'transformer_dim': 256,     #size of every vector inside the Transformer (must be divisible by n_heads)
  'n_heads': 4,               #attention heads per layer
  'ff_dim': 512,              #size of the hidden layer in each feed-forward block
  'dropout': 0.1,             #fraction of values randomly zeroed during training, against overfitting

  ### training ###
  'n_epochs': 10,
  'learning_rate': 0.001,
  'grad_clip': 1.0,           #max gradient size - stops the "exploding gradients" RNNs are prone to
  'checkpoint_path': 'checkpoints/seq2seq_best.pt',

  ### translating ###
  'beam_width': 3,            #1 = greedy decoding
  'max_output_words': 40,     #max tokens generated per sentence (subword pieces count separately)
  'norm_method': 'avg',       #'none', 'avg' or 'norm' - see normalization_scores()
  'bleu_sentences': 500,      #how many sentences BLEU is computed on (more = more reliable, slower)
}

#sentences printed after every epoch, to watch the translations improve
EXAMPLE_SENTENCES = ['I have to go to sleep.', 'Where are we going?', 'Tom is my friend.']

## 2. Preparing the Data: From Sentence Pairs to Batches

Steps:
1. **Load** the Tatoeba sentence pairs (`quoting=csv.QUOTE_NONE` so a `"` inside a sentence doesn't swallow the lines after it).
2. **Sample** a fraction (same `random_state=42` as `get_sample` in the preprocessing notebook).
3. **Tokenize**: lowercase, then split into words and punctuation.
4. **Filter** out very long sentences.
5. **Split** into train / validation / test **by English sentence**. Many English sentences have several Italian translations. If one translation is in training and another is in test, the test score is too optimistic, so every translation of a sentence stays in the same split.
6. **Segmentation**: each language's words are turned into the tokens the model sees, using the method chosen in `segmentation` (see below).
7. **Vocabulary from training data only.** The model shouldn't know any test words. Anything it can't represent becomes `<unk>`, just like in real use.
8. **Batches**: each batch is padded only to its own longest sentence, not the longest one in the whole dataset.

In [3]:
#special tokens - they always get the same index in both vocabularies
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


#accept either the .zip or the unzipped .tsv, whichever is in the folder
def find_data_file(path):
  for candidate in [path, path.replace('.zip', ''), path + '.zip']:
    if os.path.exists(candidate):
      return candidate
  raise FileNotFoundError('Could not find ' + path + ' (current folder: ' + os.getcwd() + ')')


#load all sentence pairs -> DataFrame with columns eng_sentence, it_sentence
def load_sentence_pairs(path):
  path = find_data_file(path)
  read_options = dict(sep='\t', header=None, quoting=csv.QUOTE_NONE)

  if path.endswith('.zip'):
    with zipfile.ZipFile(path) as zip_file:
      #skip the hidden "__MACOSX" copy that macOS adds to zip files
      tsv_name = [name for name in zip_file.namelist() if name.endswith('.tsv') and not name.startswith('__MACOSX')][0]
      with zip_file.open(tsv_name) as tsv_file:
        df = pd.read_csv(tsv_file, **read_options)
  else:
    df = pd.read_csv(path, **read_options)

  df.columns = ['eng_id', 'eng_sentence', 'it_id', 'it_sentence']
  df = df.dropna(subset=['eng_sentence', 'it_sentence'])
  return df[['eng_sentence', 'it_sentence']].reset_index(drop=True)


#"Let's go!" -> ['let', "'", 's', 'go', '!']
def tokenize(sentence):
  return re.findall(r"\b\w+\b|[^\w\s]", sentence.lower())


#['devo', 'andare', '.'] -> "devo andare."   (only for displaying results)
def join_tokens(tokens):
  text = ' '.join(tokens)
  text = re.sub(r"\s+([.,!?;:])", r"\1", text)   #no space before punctuation
  text = re.sub(r"\s*'\s*", "'", text)           #l ' uomo -> l'uomo
  return text

### Segmentation methods

Italian builds many forms from one stem. The verb *andare* ("to go") alone appears in the training data as *andiamo, andate, andremo, andrebbe, andarcene, andassimo, …*. As whole words, each form is a separate entry the model has to learn on its own, and about half of all Italian word forms appear only once in the training data. Splitting words into smaller, reusable units lets the model learn a stem once and an ending once.

Four ways of turning words into tokens are implemented, and each language can use a different one (`CONFIG['segmentation']`):

| method | *andremo* becomes | how the split is decided | extra libraries |
|---|---|---|---|
| `'words'` | `andremo` | no split | none |
| `'bpe'` | e.g. `andr@@ emo` | the most frequent letter sequences in the training data | none |
| `'stem+bpe'` | `andr@@ emo` | grammar rules (Snowball stemmer) cut off the ending, then BPE | nltk |
| `'lemma+tags'` | `andremo`, but *parliamo* → `parlare #VERB:…Person=1…` | a trained tagger gives the dictionary form and the grammar | spaCy + language model |

`@@` marks a piece that continues into the next one, so `andr@@ emo` joins back into *andremo*.

All four are classes with the same methods, so the rest of the notebook doesn't need to know which one is in use:
- `fit(word_lists)`: learn from the training sentences
- `to_tokens(words)`: one sentence of words → the tokens the model sees
- `to_words(tokens)`: tokens the model produced → whole words, for display and BLEU
- `get_state()`: what a checkpoint needs to rebuild the segmenter

From here on, the model sections say "word" for simplicity. Read it as *token*: a whole word, a subword piece, a lemma or a grammar tag, depending on the method.

#### Byte-pair encoding (BPE)

**Learning the pieces** (on the training sentences only):
1. Write every word as characters, marking the last one with `</w>`: *casa* → `c a s a</w>`. The marker lets BPE tell a word *ending* (`o</w>`) apart from the same letters in the middle of a word.
2. Count every pair of neighbouring symbols, weighted by how often the word occurs.
3. Merge the most frequent pair everywhere (`a n` → `an`) and record the merge.
4. Repeat for `bpe_merges` merges, or until no pair occurs at least twice.

**Splitting a word** with the learned merges: start from characters and keep applying the earliest-learned merge that fits until none do. Common words end up as a single piece (*il, è, casa*), and rare words become a few familiar pieces.

BPE only looks at frequency, not grammar. A common form like *andiamo* stays whole, so the model never sees that it shares *and-* with *andrebbero*.

In [4]:
END_OF_WORD = '</w>'      #marks the last symbol of a word while BPE is learning
CONTINUES = '@@'          #marks a piece that is followed by more of the same word: 'and@@ iamo' = 'andiamo'


#merge every occurrence of `pair` in a word: ('a','n','d','o</w>') + ('a','n') -> ('an','d','o</w>')
def merge_pair(symbols, pair):
  merged = []
  i = 0
  while i < len(symbols):
    if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
      merged.append(symbols[i] + symbols[i + 1])
      i += 2
    else:
      merged.append(symbols[i])
      i += 1
  return tuple(merged)


#'casa' -> ('c', 'a', 's', 'a</w>')
def word_to_symbols(word):
  return tuple(word[:-1]) + (word[-1] + END_OF_WORD,)


#learn merges from the training sentences (lists of words) -> list of pairs, most frequent first
def learn_bpe(word_lists, n_merges, min_pair_count=2):
  word_counts = Counter(word for words in word_lists for word in words)
  splits = {word: word_to_symbols(word) for word in word_counts}     #current symbols of every distinct word

  #how often each neighbouring pair occurs (weighted by word count), and which words contain it
  pair_counts = Counter()
  words_with_pair = defaultdict(set)
  for word, symbols in splits.items():
    for pair in zip(symbols, symbols[1:]):
      pair_counts[pair] += word_counts[word]
      words_with_pair[pair].add(word)

  merges = []
  for _ in range(n_merges):
    if not pair_counts:
      break
    best_pair = max(pair_counts, key=pair_counts.get)
    if pair_counts[best_pair] < min_pair_count:     #a pair seen only once won't help with new words
      break
    merges.append(best_pair)

    #only words that contain the pair change: remove their old pairs from the counts, merge, add the new pairs
    for word in list(words_with_pair[best_pair]):
      old_symbols = splits[word]
      new_symbols = merge_pair(old_symbols, best_pair)

      for pair in zip(old_symbols, old_symbols[1:]):
        pair_counts[pair] -= word_counts[word]
        if pair_counts[pair] <= 0:
          del pair_counts[pair]
        words_with_pair[pair].discard(word)
      for pair in zip(new_symbols, new_symbols[1:]):
        pair_counts[pair] += word_counts[word]
        words_with_pair[pair].add(word)

      splits[word] = new_symbols

  return merges


#splits words into pieces using learned merges
class BPE:
  def __init__(self, merges):
    self.merges = [tuple(pair) for pair in merges]
    self.merge_rank = {pair: rank for rank, pair in enumerate(self.merges)}   #lower rank = learned earlier
    self.cache = {}                                                           #word -> pieces, so each word is split only once

  #'andiamo' -> ['and@@', 'iamo']
  def split_word(self, word):
    if word not in self.cache:
      symbols = word_to_symbols(word)

      #keep applying the earliest-learned merge that fits, until none do
      while len(symbols) > 1:
        known_pairs = [pair for pair in zip(symbols, symbols[1:]) if pair in self.merge_rank]
        if not known_pairs:
          break
        symbols = merge_pair(symbols, min(known_pairs, key=self.merge_rank.get))

      #last piece: drop '</w>'.  Every other piece: add '@@' (the word continues)
      self.cache[word] = [s[:-len(END_OF_WORD)] if s.endswith(END_OF_WORD) else s + CONTINUES for s in symbols]
    return self.cache[word]

  #['andiamo', 'a', 'casa'] -> ['and@@', 'iamo', 'a', 'casa']
  def tokenize(self, words):
    return [piece for word in words for piece in self.split_word(word)]


#['and@@', 'iamo', 'a', 'casa'] -> ['andiamo', 'a', 'casa']   (does nothing to whole words)
def merge_subwords(pieces):
  words, current = [], ''
  for piece in pieces:
    if piece.endswith(CONTINUES):
      current += piece[:-len(CONTINUES)]
    else:
      words.append(current + piece)
      current = ''
  if current:            #the model stopped in the middle of a word
    words.append(current)
  return words

In [5]:
#small demo: learn 10 merges from a few forms of "andare" and see what BPE does
demo_sentences = [['andiamo', 'andate', 'andare'], ['andiamo', 'andremo', 'andare'], ['andate', 'andrete', 'mangiare']]
demo_merges = learn_bpe(demo_sentences, n_merges=10)
print('Learned merges, in order:', demo_merges)

demo_bpe = BPE(demo_merges)
for word in ['andiamo', 'andremo', 'andranno', 'mangiamo']:
  print('  {:10s} -> {}'.format(word, demo_bpe.split_word(word)))
print('Joined back:', merge_subwords(demo_bpe.tokenize(['andranno', 'a', 'casa'])))

Learned merges, in order: [('a', 'n'), ('an', 'd'), ('and', 'a'), ('i', 'a'), ('m', 'o</w>'), ('t', 'e</w>'), ('r', 'e</w>'), ('r', 'e'), ('and', 'ia'), ('anda', 'te</w>')]
  andiamo    -> ['andia@@', 'mo']
  andremo    -> ['and@@', 're@@', 'mo']
  andranno   -> ['and@@', 'r@@', 'an@@', 'n@@', 'o']
  mangiamo   -> ['m@@', 'an@@', 'g@@', 'ia@@', 'mo']
Joined back: ['andranno', 'a', 'casa']


#### Stemmer + BPE

A **stemmer** removes word endings with hand-written grammar rules. The Snowball stemmer (in NLTK) has rules for Italian: *andremo* → stem *andr*, *parliamo* → *parl*, *case* → *cas*. Here the word is cut into **stem + ending** (*andr* + *emo*), and BPE then runs on those parts, so a piece never crosses the stem/ending boundary.

Unlike plain BPE, the split follows the grammar even for common words: *andiamo* → `and@@ iamo`. Stemmers are rough, though: irregular forms (*vado*, *è*) aren't recognized, and the cut can land in slightly odd places. Words the stemmer doesn't change, or changes in a way that isn't a simple cut, stay whole.

In [6]:
#cuts a word into stem + ending with the Snowball stemmer: 'andiamo' -> ['and', 'iamo'],  'il' -> ['il']
class StemSplitter:
  def __init__(self, language):
    from nltk.stem.snowball import SnowballStemmer    #imported here, so nltk is only needed when this method is used
    self.stemmer = SnowballStemmer(language)
    self.cache = {}

  def split(self, word):
    if word not in self.cache:
      stem = self.stemmer.stem(word)
      if stem and stem != word and word.startswith(stem):
        self.cache[word] = [stem, word[len(stem):]]
      else:
        self.cache[word] = [word]
    return self.cache[word]

#### Lemma + grammar tags

A trained **tagger** (spaCy) reads the whole sentence and gives each word its **lemma** (dictionary form) and its **grammar features**:

*parliamo* → lemma `parlare` + tag `#VERB:Mood=Ind|Number=Plur|Person=1|Tense=Pres|VerbForm=Fin`

- A word that is already in dictionary form (*casa*, *il*, *.*) stays one token with no tag.
- Every other word becomes two tokens: its lemma and its tag. The model learns *what* to say (the lemma) separately from *which form* to use (the tag). Forms of *parlare* share the token `parlare`, and all first-person-plural present verbs share one tag.

Because the model writes lemmas and tags, its output has to be turned back into real words. This step is called **reinflection**. A table built from the training data maps each (lemma, tag) pair to the word it most often came from: (`parlare`, `#VERB:…Person=1…Pres…`) → *parliamo*. If the model produces a combination never seen in training, it falls back to the lemma.

**Limitations** (measured on the 5% sample with the small Italian model `it_core_news_sm`):
- The lemmatizer misses some forms and leaves them whole, including *andremo* and the irregular *vado*. A larger model (`it_core_news_md` / `_lg`) may do better.
- Reinflection has a ceiling. Even if the translation model's output were perfect, only about 97% of test words come back exactly. Some differences aren't in the tag at all (*gli* vs *i* depends on the next word's first letter), and others come from tagger mistakes (*dovevo* → *devo*).
- Tagging the training sentences takes about a minute.

The tagger runs on the same word split as everything else, so lemmas and tags line up with the words one to one. It needs spaCy and a language model:
- Italian: `pip install spacy` then `python -m spacy download it_core_news_sm`
- English: also `python -m spacy download en_core_web_sm`

In [7]:
SPACY_MODELS = {'italian': 'it_core_news_sm', 'english': 'en_core_web_sm'}
TAG_PREFIX = '#'      #grammar tags start with '#' so they can't be confused with words


#one sentence of words -> (lemma, tag) for every word, using spaCy:
#['andremo', 'a', 'casa'] -> [('andare', '#VERB:Mood=Ind|...|Tense=Fut|...'), ('a', '#ADP:'), ('casa', '#NOUN:Gender=Fem|Number=Sing')]
class LemmaTagger:
  def __init__(self, language):
    import spacy                       #imported here, so spaCy is only needed when this method is used
    from spacy.tokens import Doc
    self.Doc = Doc
    self.nlp = spacy.load(SPACY_MODELS[language], exclude=['parser', 'ner'])   #only the parts that tag and lemmatize
    self.cache = {}

  def analyze(self, words):
    key = tuple(words)
    if key not in self.cache:
      #build the spaCy document from our own words, so the output lines up with them one to one
      doc = self.Doc(self.nlp.vocab, words=list(words))
      for _, component in self.nlp.pipeline:
        doc = component(doc)
      self.cache[key] = [(token.lemma_.lower() or token.text, TAG_PREFIX + token.pos_ + ':' + str(token.morph)) for token in doc]
    return self.cache[key]

In [8]:
##### The four segmentation methods - all with the same methods: fit, to_tokens, to_words, get_state #####

#'andremo' -> ['andremo']
class WordSegmenter:
  method = 'words'

  def __init__(self, language, n_merges=None, state=None):
    pass

  def fit(self, word_lists):
    return self

  def to_tokens(self, words):
    return list(words)

  def to_words(self, tokens):
    return list(tokens)

  def get_state(self):
    return None


#'andrebbero' -> ['and@@', 'r@@', 'ebbero']
class BPESegmenter:
  method = 'bpe'

  def __init__(self, language, n_merges=4000, state=None):
    self.n_merges = n_merges
    self.bpe = BPE(state) if state is not None else None    #state = the learned merges (when loading a checkpoint)

  def fit(self, word_lists):
    self.bpe = BPE(learn_bpe(word_lists, self.n_merges))
    return self

  def to_tokens(self, words):
    return self.bpe.tokenize(words)

  def to_words(self, tokens):
    return merge_subwords(tokens)

  def get_state(self):
    return self.bpe.merges


#'andiamo' -> stem 'and' + ending 'iamo' -> BPE on each part -> ['and@@', 'iamo']
class StemBPESegmenter:
  method = 'stem+bpe'

  def __init__(self, language, n_merges=4000, state=None):
    self.stem_splitter = StemSplitter(language)
    self.n_merges = n_merges
    self.bpe = BPE(state) if state is not None else None

  #['andiamo', 'a', 'casa'] -> ['and', 'iamo', 'a', 'cas', 'a']
  def to_parts(self, words):
    return [part for word in words for part in self.stem_splitter.split(word)]

  #BPE learns from stems and endings instead of whole words
  def fit(self, word_lists):
    self.bpe = BPE(learn_bpe([self.to_parts(words) for words in word_lists], self.n_merges))
    return self

  def to_tokens(self, words):
    tokens = []
    for word in words:
      parts = self.stem_splitter.split(word)
      for i, part in enumerate(parts):
        pieces = self.bpe.split_word(part)
        if i < len(parts) - 1:            #the stem is followed by its ending, so its last piece gets '@@' too
          pieces = pieces[:-1] + [pieces[-1] + CONTINUES]
        tokens.extend(pieces)
    return tokens

  def to_words(self, tokens):
    return merge_subwords(tokens)

  def get_state(self):
    return self.bpe.merges


#'andremo' -> ['andare', '#VERB:Mood=Ind|Number=Plur|Person=1|Tense=Fut|VerbForm=Fin'],  'casa' -> ['casa']
class LemmaTagSegmenter:
  method = 'lemma+tags'

  def __init__(self, language, n_merges=None, state=None):
    self.language = language
    self.tagger = None                    #spaCy is loaded the first time it's needed
    #reinflection table: (lemma, tag) -> word.  state = the table as a list of [lemma, tag, word] (from a checkpoint)
    self.reinflect_table = {(lemma, tag): word for lemma, tag, word in (state or [])}

  def get_tagger(self):
    if self.tagger is None:
      self.tagger = LemmaTagger(self.language)
    return self.tagger

  def to_tokens(self, words):
    tokens = []
    for word, (lemma, tag) in zip(words, self.get_tagger().analyze(words)):
      if lemma == word:
        tokens.append(word)               #already the dictionary form - no tag needed
      else:
        tokens.extend([lemma, tag])
    return tokens

  #build the reinflection table: for every (lemma, tag), the word it most often came from in the training data
  def fit(self, word_lists):
    counts = Counter()
    for words in word_lists:
      for word, (lemma, tag) in zip(words, self.get_tagger().analyze(words)):
        if lemma != word:
          counts[(lemma, tag, word)] += 1
    for (lemma, tag, word), _ in counts.most_common():      #most common first, so setdefault keeps the most common word
      self.reinflect_table.setdefault((lemma, tag), word)
    return self

  #['andare', '#VERB:...Tense=Fut...', 'a', 'casa'] -> ['andremo', 'a', 'casa']
  def to_words(self, tokens):
    words = []
    for token in tokens:
      is_tag = token.startswith(TAG_PREFIX) and ':' in token
      if not is_tag:
        words.append(token)
      elif words:                         #a tag belongs to the word right before it
        lemma = words[-1]
        words[-1] = self.reinflect_table.get((lemma, token), lemma)   #unseen combination -> keep the lemma
    return words

  def get_state(self):
    return [[lemma, tag, word] for (lemma, tag), word in self.reinflect_table.items()]


SEGMENTERS = {'words': WordSegmenter, 'bpe': BPESegmenter, 'stem+bpe': StemBPESegmenter, 'lemma+tags': LemmaTagSegmenter}


#create the segmenter chosen in the config for one language. state = saved state from a checkpoint (or None)
def make_segmenter(config, language, state=None):
  method = config['segmentation'][language]
  if method not in SEGMENTERS:
    raise ValueError('unknown segmentation method {!r} for {} - choose one of {}'.format(method, language, list(SEGMENTERS)))
  return SEGMENTERS[method](language, n_merges=config['bpe_merges'], state=state)

In [9]:
#split by English sentence so all translations of a sentence land in the same split
def split_data(df, val_frac, test_frac, seed=SEED):
  english_sentences = list(df['eng_sentence'].unique())
  random.Random(seed).shuffle(english_sentences)

  n_test = int(len(english_sentences) * test_frac)
  n_val = int(len(english_sentences) * val_frac)
  test_sentences = set(english_sentences[:n_test])
  val_sentences = set(english_sentences[n_test:n_test + n_val])

  is_test = df['eng_sentence'].isin(test_sentences)
  is_val = df['eng_sentence'].isin(val_sentences)

  train_df = df[~is_test & ~is_val].reset_index(drop=True)
  val_df = df[is_val].reset_index(drop=True)
  test_df = df[is_test].reset_index(drop=True)
  return train_df, val_df, test_df


#maps tokens <-> numbers for one language. Index 0-3 are always <pad>, <sos>, <eos>, <unk>
#The segmenter decides what a token is (whole word, subword piece, lemma or grammar tag)
class Vocabulary:
  def __init__(self, tokens, segmenter):
    self.index_to_word = SPECIAL_TOKENS + [t for t in tokens if t not in SPECIAL_TOKENS]
    self.word_to_index = {word: index for index, word in enumerate(self.index_to_word)}
    self.segmenter = segmenter

  def __len__(self):
    return len(self.index_to_word)

  #words -> the tokens the model sees
  def to_tokens(self, words):
    return self.segmenter.to_tokens(words)

  #['devo', 'andare'] -> [1, 523, 87, 2]   (adds <sos> and <eos>, unknown tokens -> <unk>)
  def encode(self, tokens):
    return [SOS_IDX] + [self.word_to_index.get(token, UNK_IDX) for token in tokens] + [EOS_IDX]

  #[1, 523, 87, 2, 0, 0] -> ['devo', 'andare']   (stops at <eos>, drops <sos>/<pad>)
  def decode(self, indices):
    tokens = []
    for index in indices:
      if index == EOS_IDX:
        break
      if index not in (PAD_IDX, SOS_IDX):
        tokens.append(self.index_to_word[index])
    return tokens

  #"Let's go!" -> indices, all steps in one call
  def encode_sentence(self, sentence):
    return self.encode(self.to_tokens(tokenize(sentence)))

  #indices -> whole words (pieces joined back, lemmas + tags turned back into words)
  def decode_to_words(self, indices):
    return self.segmenter.to_words(self.decode(indices))


#keep tokens that appear at least min_count times in the training sentences
#  word_lists: training sentences as lists of words,  segmenter: already fitted on those sentences
def build_vocabulary(word_lists, segmenter, min_count=1):
  counts = Counter(token for words in word_lists for token in segmenter.to_tokens(words))
  tokens = sorted(token for token, count in counts.items() if count >= min_count)
  return Vocabulary(tokens, segmenter)

In [10]:
#holds every sentence pair as two tensors of token indices
class TranslationDataset(Dataset):
  def __init__(self, df, eng_vocab, it_vocab):
    self.pairs = [(torch.tensor(eng_vocab.encode(eng_vocab.to_tokens(eng_words))),
                   torch.tensor(it_vocab.encode(it_vocab.to_tokens(it_words))))
                  for eng_words, it_words in zip(df['eng_words'], df['it_words'])]

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, index):
    return self.pairs[index]


#the DataLoader calls this to combine sentences into one batch
#pads only to the longest sentence in *this* batch -> (B, S) and (B, T)
def pad_batch(batch):
  eng_sentences, it_sentences = zip(*batch)
  eng_batch = pad_sequence(eng_sentences, batch_first=True, padding_value=PAD_IDX)
  it_batch = pad_sequence(it_sentences, batch_first=True, padding_value=PAD_IDX)
  return eng_batch, it_batch


#runs all the data steps above and returns everything the model needs
def prepare_data(config):
  #1) load + 2) sample
  df = load_sentence_pairs(config['data_file'])
  print('All sentence pairs:     {:,}'.format(len(df)))
  df = df.sample(frac=config['sample_frac'], random_state=42).reset_index(drop=True)
  print('Sampled pairs:          {:,}'.format(len(df)))

  #3) tokenize
  df['eng_words'] = df['eng_sentence'].apply(tokenize)
  df['it_words'] = df['it_sentence'].apply(tokenize)

  #4) filter out empty and very long sentences
  eng_len = df['eng_words'].str.len()
  it_len = df['it_words'].str.len()
  max_len = config['max_sentence_words']
  df = df[(eng_len > 0) & (it_len > 0) & (eng_len <= max_len) & (it_len <= max_len)].reset_index(drop=True)
  print('After length filter:    {:,}'.format(len(df)))

  #5) split
  train_df, val_df, test_df = split_data(df, config['val_frac'], config['test_frac'])
  print('Train / val / test:     {:,} / {:,} / {:,}'.format(len(train_df), len(val_df), len(test_df)))

  #6) segmentation + 7) vocabularies, learned from the training data only
  eng_segmenter = make_segmenter(config, 'english').fit(train_df['eng_words'])
  it_segmenter = make_segmenter(config, 'italian').fit(train_df['it_words'])

  eng_vocab = build_vocabulary(train_df['eng_words'], eng_segmenter, config['min_word_count'])
  it_vocab = build_vocabulary(train_df['it_words'], it_segmenter, config['min_word_count'])
  print('English vocab:          {:,} tokens ({})'.format(len(eng_vocab), eng_segmenter.method))
  print('Italian vocab:          {:,} tokens ({})'.format(len(it_vocab), it_segmenter.method))

  #8) batches
  batch_size = config['batch_size']
  train_loader = DataLoader(TranslationDataset(train_df, eng_vocab, it_vocab), batch_size=batch_size, shuffle=True, collate_fn=pad_batch)
  val_loader = DataLoader(TranslationDataset(val_df, eng_vocab, it_vocab), batch_size=batch_size, shuffle=False, collate_fn=pad_batch)
  test_loader = DataLoader(TranslationDataset(test_df, eng_vocab, it_vocab), batch_size=batch_size, shuffle=False, collate_fn=pad_batch)

  return train_df, val_df, test_df, eng_vocab, it_vocab, train_loader, val_loader, test_loader

## 3. Building blocks

The embedding, RNN and LSTM nodes are written from scratch instead of using `nn.RNN` / `nn.LSTM`, so each equation is visible in the code.

A few design choices:
- **Weight initialization.** Weights drawn from `torch.randn` (standard deviation 1) push sigmoid/tanh to their flat ends (near 0 or ±1). There the gradients are almost zero and nothing learns. `new_weight` uses Xavier initialization instead, which scales the values to the size of the matrix.
- **`tanh` in the RNN cell.** tanh is centred on 0, while sigmoid only outputs 0 to 1 and would shift every hidden state positive.
- **One interface for both cells:** `forward(input, prev_hidden, prev_cell) -> (hidden, cell)`. The RNN cell returns `cell=None`, so the encoder and decoder work with either cell type without special cases.
- **Batch first.** Everything is written as `x @ W.T`, so the batch is always the first dimension and the bias broadcasts automatically.

In [11]:
#a learnable weight matrix with Xavier/Glorot initialization (values scaled to the matrix size)
def new_weight(n_rows, n_cols):
  weight = torch.empty(n_rows, n_cols)
  nn.init.xavier_uniform_(weight)
  return nn.Parameter(weight)


#looks up a learned vector for every word index
#input: (B, S) word indices  ->  output: (B, S, E)
class Embedding_Node(nn.Module):
  def __init__(self, vocab_size, embed_dim):
    super(Embedding_Node, self).__init__()
    self.W_e = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.1)

  def forward(self, word_indices):
    return self.W_e[word_indices]

In [12]:
#one step of a plain RNN:   h_t = tanh(x_t·Uᵀ + h_(t-1)·Wᵀ + b)
#input: (B, input_dim), prev_hidden: (B, H)  ->  hidden: (B, H), cell: None
class RNN_Node(nn.Module):
  def __init__(self, hidden_dim, input_dim):
    super(RNN_Node, self).__init__()
    self.layer_weight = new_weight(hidden_dim, input_dim)    #U - applied to the input
    self.hidden_weight = new_weight(hidden_dim, hidden_dim)  #W - applied to the previous hidden state
    self.bias = nn.Parameter(torch.zeros(hidden_dim))

  #prev_cell is ignored - it's only there so RNN and LSTM nodes are called the same way
  def forward(self, input, prev_hidden, prev_cell=None):
    hidden = torch.tanh(input @ self.layer_weight.T + prev_hidden @ self.hidden_weight.T + self.bias)
    return hidden, None

In [13]:
#one step of an LSTM. Each gate is a sigmoid between 0 (block) and 1 (let through):
#   forget gate f: how much of the old cell memory to keep
#   input gate  i: how much of the new candidate memory to write
#   output gate o: how much of the memory to show as the hidden state
#input: (B, input_dim), prev_hidden: (B, H), prev_cell: (B, H)  ->  hidden (B, H), cell (B, H)
class LSTM_Node(nn.Module):
  def __init__(self, hidden_dim, input_dim):
    super(LSTM_Node, self).__init__()

    #forget weight / prev layer forget weight / forget bias
    #(forget bias starts at 1 so the cell remembers by default at the start of training - a standard trick)
    self.W_f, self.U_f, self.b_f = new_weight(hidden_dim, hidden_dim), new_weight(hidden_dim, input_dim), nn.Parameter(torch.ones(hidden_dim))
    #input weight / prev layer input weight / input bias
    self.W_i, self.U_i, self.b_i = new_weight(hidden_dim, hidden_dim), new_weight(hidden_dim, input_dim), nn.Parameter(torch.zeros(hidden_dim))
    #output weight / prev layer output weight / output bias
    self.W_o, self.U_o, self.b_o = new_weight(hidden_dim, hidden_dim), new_weight(hidden_dim, input_dim), nn.Parameter(torch.zeros(hidden_dim))
    #cell weight / prev layer cell weight / cell bias
    self.W_c, self.U_c, self.b_c = new_weight(hidden_dim, hidden_dim), new_weight(hidden_dim, input_dim), nn.Parameter(torch.zeros(hidden_dim))

  def forward(self, input, prev_hidden, prev_cell):
    #same pattern for every gate: input·Uᵀ + prev_hidden·Wᵀ + b
    def combine(W, U, b):
      return input @ U.T + prev_hidden @ W.T + b

    f_t = torch.sigmoid(combine(self.W_f, self.U_f, self.b_f))    #forget
    i_t = torch.sigmoid(combine(self.W_i, self.U_i, self.b_i))    #input
    o_t = torch.sigmoid(combine(self.W_o, self.U_o, self.b_o))    #output
    nc_t = torch.tanh(combine(self.W_c, self.U_c, self.b_c))      #new cell candidate

    c_t = f_t * prev_cell + i_t * nc_t    #cell: keep some old memory + write some new
    h_t = o_t * torch.tanh(c_t)           #hidden: what this step outputs
    return h_t, c_t


NODE_CLASSES = {'rnn': RNN_Node, 'lstm': LSTM_Node}

## 4. Encoder

Reads the English sentence one word at a time and keeps the top layer's hidden state at every word, which attention uses later.

**Padding.** Short sentences in a batch are padded with `<pad>` at the end. If the encoder simply kept running, the final state would describe "sentence + padding". Instead, at padded positions each layer **keeps its previous state** (`torch.where`), so the final state is the state after the last *real* word. A sentence gets the same summary whether it's alone or in a padded batch.

**Bidirectional option** (`bidirectional: True`). A normal encoder reads left to right, so its state at a word only knows the words *before* it. A bidirectional encoder adds a second stack of nodes that reads the sentence **right to left**, and the two states at each word are put side by side. Each word's vector then describes both its left and right context. That helps attention, since the meaning of *bank* or *runs* often depends on words that come later. The cost is twice the encoder work, and the encoder's vectors are twice as long (`output_dim = 2 × H`), which the decoder's attention and the starting-state layer take into account. Padding is handled the same way: reading right to left, the backward stack stays at its starting state through the padding and begins at the last real word.

The Transformer (section 12) doesn't need this option: self-attention already lets every word see the whole sentence in both directions.

**State storage.** States are kept in plain Python lists and replaced (`hidden[layer] = ...`) rather than written into one big tensor in place. PyTorch's gradient computation can fail when a tensor it still needs is overwritten.

In [14]:
#inputs: (B, S) English word indices
#returns (O = output_dim = H, or 2H if bidirectional):
#  outputs:      (B, S, O) top-layer hidden state at every word -> used by attention
#  final_hidden: list with one (B, O) tensor per layer - state after the last real word
#  final_cell:   same for the LSTM cell state (list of None for an RNN)
class EncoderNN(nn.Module):
  def __init__(self, node_type, n_layers, hidden_dim, embed_dim, vocab_size, bidirectional=False):
    super(EncoderNN, self).__init__()
    self.node_type = node_type
    self.n_layers = n_layers
    self.hidden_dim = hidden_dim
    self.bidirectional = bidirectional
    self.output_dim = hidden_dim * (2 if bidirectional else 1)   #forward and backward states side by side

    self.embedding_node = Embedding_Node(vocab_size, embed_dim)

    #first layer reads word embeddings (size E), every layer above reads the layer below it (size H)
    node_class = NODE_CLASSES[node_type]
    def make_stack():
      return nn.ModuleList([node_class(hidden_dim, embed_dim)] +
                           [node_class(hidden_dim, hidden_dim) for _ in range(n_layers - 1)])

    self.nodes = make_stack()                    #reads left to right
    if bidirectional:
      self.backward_nodes = make_stack()         #reads right to left

  #all-zero starting states, one per layer
  def init_states(self, batch_size, device):
    hidden = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.n_layers)]
    if self.node_type == 'lstm':
      cell = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.n_layers)]
    else:
      cell = [None] * self.n_layers
    return hidden, cell

  #run one stack of nodes over the sentence, visiting the word positions in the given order
  #returns outputs (B, S, H) stored at each word's own position, and the final states
  def run_direction(self, nodes, embedded, inputs, positions):
    batch_size, seq_len = inputs.shape
    hidden, cell = self.init_states(batch_size, inputs.device)
    top_layer_outputs = [None] * seq_len

    for word_i in positions:
      layer_input = embedded[:, word_i]                             # (B, E) - this word for every sentence
      is_real_word = (inputs[:, word_i] != PAD_IDX).unsqueeze(1)    # (B, 1) - False where it's padding

      for layer, node in enumerate(nodes):
        new_hidden, new_cell = node(layer_input, hidden[layer], cell[layer])

        #padding: keep the old state, so padding doesn't change the sentence summary
        hidden[layer] = torch.where(is_real_word, new_hidden, hidden[layer])
        if new_cell is not None:
          cell[layer] = torch.where(is_real_word, new_cell, cell[layer])

        #the next layer up reads this layer's hidden state
        layer_input = hidden[layer]

      top_layer_outputs[word_i] = hidden[-1]

    return torch.stack(top_layer_outputs, dim=1), hidden, cell

  def forward(self, inputs):
    seq_len = inputs.shape[1]
    embedded = self.embedding_node(inputs)                          # (B, S, E)

    #left to right
    outputs, hidden, cell = self.run_direction(self.nodes, embedded, inputs, range(seq_len))

    #right to left, then put both directions side by side at every word and in every final state
    if self.bidirectional:
      back_outputs, back_hidden, back_cell = self.run_direction(self.backward_nodes, embedded, inputs, reversed(range(seq_len)))
      outputs = torch.cat([outputs, back_outputs], dim=2)                                   # (B, S, 2H)
      hidden = [torch.cat([fwd, back], dim=1) for fwd, back in zip(hidden, back_hidden)]   # each (B, 2H)
      cell = [None if fwd is None else torch.cat([fwd, back], dim=1) for fwd, back in zip(cell, back_cell)]

    return outputs, hidden, cell

## 5. Attention

At every Italian word, the decoder asks: *which English words should I look at right now?*

- **query** = the decoder's current hidden state ("what am I looking for?")
- **keys** = one vector per English word ("what does each word offer?")
- **values** = one vector per English word ("what do I take from it?")

score(query, key) → softmax → weights that sum to 1 → weighted sum of the values = **context**.

**Masking:**
1. **Padding mask.** Padded positions get a score of `-inf`, so their weight after the softmax is exactly 0.
2. **Causal mask (optional, off here).** An upper-triangle mask stops position *i* from looking at later positions. That's needed for decoder *self*-attention in a Transformer, but not for encoder–decoder attention. With a single decoder query, it would hide every English word except the first. So `causal=False` is used here. The Transformer in section 12 applies the causal mask in its decoder self-attention.

This is single-head attention. The Transformer in section 12 uses multi-head attention.

In [15]:
#query_hidden:    (B, Q, dec_H)  decoder state(s) asking the question (Q = 1 in our decoder)
#encoder_outputs: (B, S, enc_H)  one vector per English word (used for both keys and values)
#src_mask:        (B, S)         True for real words, False for padding
#returns: context (B, Q, val_dim) and attention weights (B, Q, S)
#
#sim_method - how a query and a key are compared:
#  'dot'        score = q · k
#  'scaled_dot' score = q · k / sqrt(attn_dim)      (stops scores growing with the vector size)
#  'bahdanau'   score = vᵀ · tanh(q + k)            (additive attention - a tiny neural net; q, k already projected)
#causal - ONLY for decoder self-attention (Transformer): position i can't look at positions after i.
#         Keep it False for encoder-decoder attention.
class Attention_Node(nn.Module):
  def __init__(self, enc_hidden_dim, dec_hidden_dim, attn_dim, val_dim, sim_method='scaled_dot', causal=False):
    super(Attention_Node, self).__init__()
    self.attn_dim = attn_dim
    self.sim_method = sim_method
    self.causal = causal

    #key, query, value projections
    self.W_key = nn.Linear(enc_hidden_dim, attn_dim)
    self.W_query = nn.Linear(dec_hidden_dim, attn_dim)
    self.W_value = nn.Linear(enc_hidden_dim, val_dim)

    #bahdanau / additive method needs one extra vector v to turn tanh(q + k) into a single score
    if sim_method in ('bahdanau', 'add'):
      self.V_a_bah = nn.Linear(attn_dim, 1, bias=False)
    elif sim_method not in ('dot', 'scaled_dot'):
      raise ValueError('unknown sim_method: ' + sim_method)

  def forward(self, query_hidden, encoder_outputs, src_mask=None):
    key = self.W_key(encoder_outputs)        # (B, S, A)
    value = self.W_value(encoder_outputs)    # (B, S, val_dim)
    query = self.W_query(query_hidden)       # (B, Q, A)

    #attention scores: one score for every (query, English word) pair -> (B, Q, S)
    if self.sim_method == 'dot':
      scores = query @ key.transpose(1, 2)
    elif self.sim_method == 'scaled_dot':
      scores = query @ key.transpose(1, 2) / math.sqrt(self.attn_dim)
    else:  #bahdanau: (B, Q, 1, A) + (B, 1, S, A) -> (B, Q, S, A) -> v -> (B, Q, S)
      scores = self.V_a_bah(torch.tanh(query.unsqueeze(2) + key.unsqueeze(1))).squeeze(-1)

    #padding mask: never attend to <pad>
    if src_mask is not None:
      scores = scores.masked_fill(~src_mask.unsqueeze(1), float('-inf'))

    #causal mask (Transformer decoder self-attention only): block positions after the query
    if self.causal:
      n_queries, n_keys = scores.shape[1], scores.shape[2]
      future = torch.triu(torch.ones(n_queries, n_keys, dtype=torch.bool, device=scores.device), diagonal=1)
      scores = scores.masked_fill(future, float('-inf'))

    weights = F.softmax(scores, dim=-1)      # (B, Q, S) - each row sums to 1
    context = weights @ value                # (B, Q, val_dim) - weighted sum of the values
    return context, weights

## 6. Decoder

`step()` handles **one** Italian word: previous word in, scores for the next word out.

With attention, the top hidden state and the context are combined (`attn_combine`) into an "attentional" vector, and the prediction is made from that vector. This is Luong-style attention. The combined vector is used **only for the prediction**; the recurrent hidden state carries on unchanged to the next step.

The output layer produces **logits**: raw scores, one per Italian word. `CrossEntropyLoss` (training) and `log_softmax` (beam search) turn them into probabilities.

In [16]:
class DecoderNN(nn.Module):
  def __init__(self, node_type, n_layers, hidden_dim, embed_dim, vocab_size, enc_hidden_dim,
               attention=True, attn_dim=128, sim_method='scaled_dot'):
    super(DecoderNN, self).__init__()
    self.node_type = node_type
    self.n_layers = n_layers
    self.hidden_dim = hidden_dim
    self.vocab_size = vocab_size
    self.use_attention = attention

    self.embedding_node = Embedding_Node(vocab_size, embed_dim)

    node_class = NODE_CLASSES[node_type]
    self.nodes = nn.ModuleList(
          [node_class(hidden_dim, embed_dim)] +
          [node_class(hidden_dim, hidden_dim) for _ in range(n_layers - 1)])

    if attention:
      self.attention_node = Attention_Node(enc_hidden_dim, hidden_dim, attn_dim, val_dim=enc_hidden_dim, sim_method=sim_method)
      #combine "what I'm thinking" (hidden) with "what I'm looking at" (context) back down to size H
      self.attn_combine = nn.Linear(hidden_dim + enc_hidden_dim, hidden_dim)

    #hidden state -> one score per Italian word
    self.output_layer = nn.Linear(hidden_dim, vocab_size)

  #one decoding step
  #  prev_word:       (B,)   index of the previous Italian word (<sos> at the start)
  #  hidden, cell:    lists with one (B, H) tensor per layer
  #  encoder_outputs: (B, S, enc_H),  src_mask: (B, S)
  #returns logits (B, V), new hidden list, new cell list, attention weights (B, S) or None
  def step(self, prev_word, hidden, cell, encoder_outputs, src_mask):
    layer_input = self.embedding_node(prev_word)       # (B, E)

    new_hidden, new_cell = [], []
    for layer, node in enumerate(self.nodes):
      h, c = node(layer_input, hidden[layer], cell[layer])
      new_hidden.append(h)
      new_cell.append(c)
      layer_input = h                                  #the next layer up reads this layer's output

    top_hidden = new_hidden[-1]                        # (B, H)
    attn_weights = None

    if self.use_attention:
      context, attn_weights = self.attention_node(top_hidden.unsqueeze(1), encoder_outputs, src_mask)  # (B, 1, enc_H)
      combined = torch.cat([top_hidden, context.squeeze(1)], dim=1)                                    # (B, H + enc_H)
      top_hidden = torch.tanh(self.attn_combine(combined))                                             # (B, H)
      attn_weights = attn_weights.squeeze(1)                                                           # (B, S)

    logits = self.output_layer(top_hidden)             # (B, V)
    return logits, new_hidden, new_cell, attn_weights

## 7. Seq2Seq: encoder + decoder

**Teacher forcing.** A target sentence looks like `<sos> w1 w2 … wn <eos>`. During training it's shifted by one position:
- decoder **input** = `<sos> w1 … wn` = `tgt[:, :-1]`
- decoder **target** = `w1 … wn <eos>` = `tgt[:, 1:]` (used in the loss)

At each step the decoder gets the *true* previous word and has to predict the *next* one. Without the shift, it would be fed word *t* and scored on word *t*, and it would only learn to copy its input.

**Starting state.** Every decoder layer starts from the encoder's summary of the sentence. A linear layer converts it, so the encoder and decoder can have different hidden sizes.

In [17]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super(Seq2Seq, self).__init__()
    self.encoder = encoder
    self.decoder = decoder

    #encoder and decoder can have different sizes (and a bidirectional encoder doubles its size)
    #-> linear layers convert the encoder's final state to the decoder's size
    self.hidden_transform = nn.Linear(encoder.output_dim, decoder.hidden_dim)
    self.cell_transform = nn.Linear(encoder.output_dim, decoder.hidden_dim)

  #run the encoder and build the decoder's starting state
  #src: (B, S)  ->  encoder_outputs (B, S, enc_H), src_mask (B, S), decoder hidden list, decoder cell list
  def encode(self, src):
    src_mask = src != PAD_IDX
    encoder_outputs, enc_hidden, enc_cell = self.encoder(src)

    #every decoder layer starts from the encoder's top-layer summary of the sentence
    start_hidden = torch.tanh(self.hidden_transform(enc_hidden[-1]))
    dec_hidden = [start_hidden for _ in range(self.decoder.n_layers)]

    if self.decoder.node_type == 'lstm':
      if enc_cell[-1] is not None:
        start_cell = self.cell_transform(enc_cell[-1])
      else:  #RNN encoder has no cell state -> start the LSTM decoder's cell at zero
        start_cell = torch.zeros_like(start_hidden)
      dec_cell = [start_cell for _ in range(self.decoder.n_layers)]
    else:
      dec_cell = [None] * self.decoder.n_layers

    return encoder_outputs, src_mask, dec_hidden, dec_cell

  #training forward pass with teacher forcing
  #src: (B, S), tgt: (B, T)  ->  logits (B, T-1, V): a prediction for tgt[:, 1:]
  def forward(self, src, tgt):
    encoder_outputs, src_mask, hidden, cell = self.encode(src)

    decoder_inputs = tgt[:, :-1]          #<sos> w1 ... wn   (the true previous words)
    all_logits = []
    for word_pos in range(decoder_inputs.shape[1]):
      logits, hidden, cell, _ = self.decoder.step(decoder_inputs[:, word_pos], hidden, cell, encoder_outputs, src_mask)
      all_logits.append(logits)

    return torch.stack(all_logits, dim=1)  # (B, T-1, V)


#build a fresh model from the settings ('transformer' -> section 12, 'rnn' / 'lstm' -> the classes above)
def build_model(config, eng_vocab_size, it_vocab_size):
  if config['node_type'] == 'transformer':
    return build_transformer(config, eng_vocab_size, it_vocab_size)

  encoder = EncoderNN(config['node_type'], config['enc_n_layers'], config['enc_hidden_dim'], config['embed_dim'], eng_vocab_size,
                      bidirectional=config['bidirectional'])
  decoder = DecoderNN(config['node_type'], config['dec_n_layers'], config['dec_hidden_dim'], config['embed_dim'], it_vocab_size,
                      enc_hidden_dim=encoder.output_dim, attention=config['attention'],
                      attn_dim=config['attn_dim'], sim_method=config['sim_method'])
  return Seq2Seq(encoder, decoder).to(device)

## 8. Beam search: from scores to a sentence

The model gives scores for the *next* word. To translate, we start with `<sos>` and repeatedly pick next words:
- **Greedy** (`beam_width=1`): always take the single best word.
- **Beam search**: keep the `beam_width` best partial sentences (beams), extend each one with every possible word, and keep the best `beam_width` again. A beam that produces `<eos>` is finished.

The score of a beam is the **sum of the log-probabilities** of its words. Each word adds a negative number, so longer sentences score lower. `normalization_scores` corrects for that when picking the final answer.

Beam search here translates one sentence at a time, with the live beams of that sentence stacked into a small batch. That keeps the bookkeeping easy to follow. The search starts from a **single** `<sos>` beam. Starting with `beam_width` identical beams would fill the beam with copies of the same word after the first step.

In [18]:
#Longer sentences add more negative log-probabilities, so raw scores favour short outputs.
#   'none' - raw sum (bias to shorter sentences)
#   'avg'  - average per word (no bias to long or short)
#   'norm' - Google NMT length penalty ((5 + length) / 6)^alpha - alpha decides how much to penalize length
def normalization_scores(log_prob, length, norm_method='avg', alpha=0.6):
  if norm_method == 'avg':
    return log_prob / length
  elif norm_method == 'norm':
    return log_prob / (((5 + length) / 6) ** alpha)
  else:
    return log_prob


#one partial translation: words so far (starting with <sos>), total log-probability, and decoder state
Beam = namedtuple('Beam', ['words', 'log_prob', 'hidden', 'cell'])


#translate ONE sentence. src: (S,) tensor of English word indices. beam_width=1 -> greedy decoding
#returns the best list of Italian word indices (without <sos>/<eos>)
def beam_search(model, src, beam_width=3, max_length=40, norm_method='avg', alpha=0.6):
  model.eval()
  with torch.no_grad():
    encoder_outputs, src_mask, hidden, cell = model.encode(src.unsqueeze(0).to(device))   #a batch of 1 sentence
    vocab_size = model.decoder.vocab_size

    beams = [Beam(words=[SOS_IDX], log_prob=0.0, hidden=hidden, cell=cell)]
    finished = []

    for _ in range(max_length):
      n_beams = len(beams)

      #1) run the decoder on every live beam at once (stack the beams into a batch)
      prev_words = torch.tensor([beam.words[-1] for beam in beams], device=device)
      #(the state is a list of tensors: one per layer for an RNN/LSTM, the words so far for a Transformer)
      hidden = [torch.cat([beam.hidden[i] for beam in beams]) for i in range(len(beams[0].hidden))]
      if beams[0].cell[0] is None:
        cell = beams[0].cell
      else:
        cell = [torch.cat([beam.cell[i] for beam in beams]) for i in range(len(beams[0].cell))]

      logits, new_hidden, new_cell, _ = model.decoder.step(
          prev_words, hidden, cell,
          encoder_outputs.expand(n_beams, -1, -1), src_mask.expand(n_beams, -1))

      log_probs = F.log_softmax(logits, dim=-1)       # (n_beams, V)
      log_probs[:, PAD_IDX] = float('-inf')           #never predict <pad> or <sos>
      log_probs[:, SOS_IDX] = float('-inf')

      #2) score of every (beam, next word) pair = the beam's score so far + log-prob of the new word
      beam_scores = torch.tensor([beam.log_prob for beam in beams], device=device).unsqueeze(1)   # (n_beams, 1)
      candidate_scores = (beam_scores + log_probs).flatten()                                       # (n_beams * V)

      #3) look at the best 2 x beam_width candidates, so there are still enough left if some end with <eos>
      top_scores, top_positions = candidate_scores.topk(min(2 * beam_width, candidate_scores.numel()))

      new_beams = []
      for score, position in zip(top_scores.tolist(), top_positions.tolist()):
        parent, word = divmod(position, vocab_size)    #which beam it came from, and which word was added
        candidate = Beam(words=beams[parent].words + [word],
                         log_prob=score,
                         hidden=[h[parent:parent + 1] for h in new_hidden],
                         cell=[None if c is None else c[parent:parent + 1] for c in new_cell])
        if word == EOS_IDX:
          finished.append(candidate)
        else:
          new_beams.append(candidate)
        if len(new_beams) == beam_width:
          break
      beams = new_beams

      #4) stop once we have enough finished sentences
      if len(finished) >= beam_width or not beams:
        break

    #pick the best finished sentence (or the best unfinished one if none reached <eos> in time)
    candidates = finished if finished else beams
    best = max(candidates, key=lambda beam: normalization_scores(beam.log_prob, len(beam.words) - 1, norm_method, alpha))
    return [word for word in best.words if word not in (SOS_IDX, EOS_IDX)]


#English sentence (text) -> Italian sentence (text)
def translate(model, sentence, eng_vocab, it_vocab, beam_width=3, max_length=40, norm_method='avg'):
  src = torch.tensor(eng_vocab.encode_sentence(sentence))
  it_indices = beam_search(model, src, beam_width, max_length, norm_method)
  return join_tokens(it_vocab.decode_to_words(it_indices))

## 9. Training loop

One **epoch** = one pass over all training sentences. For each batch:
1. forward pass → logits for every next word
2. **loss** = cross-entropy between logits and the true next words. `ignore_index=PAD_IDX` means padding never counts.
3. `loss.backward()` → gradients
4. **gradient clipping**: if the gradients are huge, scale them down (RNNs sometimes "explode")
5. `optimizer.step()` → update the weights (Adam)

After each epoch we measure the **validation loss** on sentences the model never trains on.
- If it goes down, the model is generalizing; save a **checkpoint**.
- If train loss keeps falling while validation loss rises, the model is **overfitting** (memorizing).

**Perplexity** = e^loss: roughly "how many words the model is choosing between" at each step. Lower is better.

In [19]:
#one pass over the data
#  with an optimizer    -> training (weights are updated)
#  without an optimizer -> evaluation (no gradients, no updates)
#returns the average loss per (non-padding) word
def run_epoch(model, loader, loss_fn, optimizer=None, grad_clip=1.0, print_every=100):
  training = optimizer is not None
  model.train(training)

  total_loss, total_words = 0.0, 0
  start = time.time()

  with torch.set_grad_enabled(training):
    for batch_i, (src, tgt) in enumerate(loader, start=1):
      src, tgt = src.to(device), tgt.to(device)

      logits = model(src, tgt)                       # (B, T-1, V)
      targets = tgt[:, 1:]                           # (B, T-1)  the true NEXT word at every position
      loss = loss_fn(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))

      if training:
        optimizer.zero_grad()                        #clear gradients from the previous batch
        loss.backward()                              #compute new gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()                             #update the weights

      #keep a running total weighted by the number of real words in the batch
      n_words = (targets != PAD_IDX).sum().item()
      total_loss += loss.item() * n_words
      total_words += n_words

      if training and print_every and batch_i % print_every == 0:
        print('   batch {}/{}  loss {:.3f}  ({:.0f}s)'.format(batch_i, len(loader), total_loss / total_words, time.time() - start))

  return total_loss / total_words


#save everything needed to use the model later (weights + settings + vocabularies + segmenter states)
def save_checkpoint(path, model, config, eng_vocab, it_vocab):
  folder = os.path.dirname(path)
  if folder:
    os.makedirs(folder, exist_ok=True)
  torch.save({'model_state': model.state_dict(),
              'config': config,
              'eng_words': eng_vocab.index_to_word,
              'it_words': it_vocab.index_to_word,
              'eng_segmenter_state': eng_vocab.segmenter.get_state(),
              'it_segmenter_state': it_vocab.segmenter.get_state()}, path)


#rebuild the model and vocabularies from a saved checkpoint
def load_checkpoint(path):
  checkpoint = torch.load(path, map_location=device)
  config = checkpoint['config']
  eng_vocab = Vocabulary(checkpoint['eng_words'], make_segmenter(config, 'english', checkpoint['eng_segmenter_state']))
  it_vocab = Vocabulary(checkpoint['it_words'], make_segmenter(config, 'italian', checkpoint['it_segmenter_state']))
  model = build_model(checkpoint['config'], len(eng_vocab), len(it_vocab))
  model.load_state_dict(checkpoint['model_state'])
  return model, eng_vocab, it_vocab

In [20]:
def train_model(model, train_loader, val_loader, config, eng_vocab, it_vocab, example_sentences=()):
  optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
  loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

  history = []
  best_val_loss = float('inf')

  for epoch in range(1, config['n_epochs'] + 1):
    start = time.time()
    print('\nEpoch {}/{}'.format(epoch, config['n_epochs']))

    train_loss = run_epoch(model, train_loader, loss_fn, optimizer, config['grad_clip'])
    val_loss = run_epoch(model, val_loader, loss_fn)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    print('   train loss {:.3f} | val loss {:.3f} | val perplexity {:.1f} | {:.1f} min'.format(
        train_loss, val_loss, math.exp(val_loss), (time.time() - start) / 60))

    #keep the version of the model that does best on the validation data
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      save_checkpoint(config['checkpoint_path'], model, config, eng_vocab, it_vocab)
      print('   saved checkpoint (best validation loss so far)')

    #watch the translations improve
    for sentence in example_sentences:
      print('   {}  ->  {}'.format(sentence, translate(model, sentence, eng_vocab, it_vocab, config['beam_width'], config['max_output_words'], config['norm_method'])))

  return pd.DataFrame(history)

## 10. Evaluation: loss and BLEU

Two kinds of held-out sentences are scored here:
- **validation** sentences pick the best epoch and the best configuration (in `Experiments.ipynb`)
- **test** sentences are used once, on the final chosen model, so its score is an honest estimate for new sentences

**BLEU** (0–100) compares the model's translation with human translations:
1. For n = 1…4: what fraction of the model's n-grams (single words, word pairs, triples, quadruples) also appear in a reference? Each reference n-gram can only be matched as many times as it appears ("clipping"), so repeating "the the the" doesn't score points.
2. Take the geometric mean of the 4 precisions.
3. **Brevity penalty**: lower the score if the translations are shorter than the references. Otherwise very short, "safe" outputs would win.

Tatoeba often has several Italian translations for one English sentence, so every translation is used as a reference. Scores of around 20–30 are reasonable for a small model like this.

In [21]:
#corpus BLEU (Papineni et al., 2002), 0-100
#  references: one list per sentence, containing one or more reference translations (token lists)
#  hypotheses: one token list per sentence (the model's translations)
def bleu_score(references, hypotheses, max_n=4):
  matches = [0] * max_n     #n-grams the model got right, for n = 1..4
  totals = [0] * max_n      #n-grams the model produced
  hyp_length, ref_length = 0, 0

  for refs, hyp in zip(references, hypotheses):
    hyp_length += len(hyp)
    #for the brevity penalty use the reference closest in length to the model's output
    ref_length += min((abs(len(ref) - len(hyp)), len(ref)) for ref in refs)[1]

    for n in range(1, max_n + 1):
      hyp_ngrams = Counter(tuple(hyp[i:i + n]) for i in range(len(hyp) - n + 1))
      #clip: an n-gram counts at most as often as it appears in any single reference
      max_ref_counts = Counter()
      for ref in refs:
        ref_ngrams = Counter(tuple(ref[i:i + n]) for i in range(len(ref) - n + 1))
        for ngram, count in ref_ngrams.items():
          max_ref_counts[ngram] = max(max_ref_counts[ngram], count)

      matches[n - 1] += sum(min(count, max_ref_counts[ngram]) for ngram, count in hyp_ngrams.items())
      totals[n - 1] += max(len(hyp) - n + 1, 0)

  if min(matches) == 0:
    return 0.0

  #geometric mean of the precisions (done with logs to avoid tiny numbers)
  log_precision = sum(math.log(m / t) for m, t in zip(matches, totals)) / max_n
  brevity_penalty = 1.0 if hyp_length > ref_length else math.exp(1 - ref_length / hyp_length)
  return 100 * brevity_penalty * math.exp(log_precision)


#translate test sentences and compute BLEU. Returns (score, table of examples)
#BLEU is always computed on whole words (subword pieces are joined back first), so scores are comparable with or without BPE
def evaluate_bleu(model, test_df, eng_vocab, it_vocab, n_sentences=500, beam_width=3, max_length=40, norm_method='avg'):
  #English sentence -> all of its Italian translations
  references_by_sentence = test_df.groupby('eng_sentence', sort=False)['it_words'].apply(list)
  references_by_sentence = references_by_sentence.iloc[:n_sentences]

  references, hypotheses, rows = [], [], []
  for eng_sentence, refs in references_by_sentence.items():
    src = torch.tensor(eng_vocab.encode_sentence(eng_sentence))
    hyp = it_vocab.decode_to_words(beam_search(model, src, beam_width, max_length, norm_method))
    references.append(refs)
    hypotheses.append(hyp)
    rows.append({'English': eng_sentence, 'Reference': join_tokens(refs[0]), 'Model': join_tokens(hyp)})

  return bleu_score(references, hypotheses), pd.DataFrame(rows)

In [22]:
#quick check that BLEU behaves: a perfect match scores 100, a completely wrong one scores 0
print(bleu_score([[['devo', 'andare', 'a', 'dormire', '.']]], [['devo', 'andare', 'a', 'dormire', '.']]))
print(bleu_score([[['devo', 'andare', 'a', 'dormire', '.']]], [['ciao', 'mondo']]))

100.0
0.0


## 11. Putting it together: `run_experiment(config)`

One call runs the whole pipeline for one configuration:
1. prepare the data (`prepare_data`)
2. build a fresh model (`build_model`)
3. train it, saving the epoch with the lowest validation loss (`train_model`)
4. reload that best epoch and compute **validation BLEU** (`evaluate_bleu`)

It returns a small dictionary of results, which `Experiments.ipynb` collects into a table. Validation **loss** can't be compared across segmentation methods, because each method predicts different tokens. Validation **BLEU** is always computed on whole words, so it can be compared, and it's what experiments are ranked by.

In [23]:
def run_experiment(config, example_sentences=EXAMPLE_SENTENCES):
  start = time.time()

  #1) data  2) model  3) training
  train_df, val_df, test_df, eng_vocab, it_vocab, train_loader, val_loader, test_loader = prepare_data(config)
  model = build_model(config, len(eng_vocab), len(it_vocab))
  history = train_model(model, train_loader, val_loader, config, eng_vocab, it_vocab, example_sentences)

  #4) the best epoch (lowest validation loss) was saved to the checkpoint -> reload it and score it with BLEU
  model, eng_vocab, it_vocab = load_checkpoint(config['checkpoint_path'])
  val_bleu, _ = evaluate_bleu(model, val_df, eng_vocab, it_vocab, n_sentences=config['bleu_sentences'],
                              beam_width=config['beam_width'], max_length=config['max_output_words'],
                              norm_method=config['norm_method'])

  best = history.loc[history['val_loss'].idxmin()]
  return {'val_bleu': round(val_bleu, 2),
          'val_loss': round(float(best['val_loss']), 3),
          'best_epoch': int(best['epoch']),
          'english_vocab': len(eng_vocab),
          'italian_vocab': len(it_vocab),
          'minutes': round((time.time() - start) / 60, 1),
          'checkpoint': config['checkpoint_path']}

## 12. Transformer: an alternative to the RNN / LSTM

Set `node_type: 'transformer'` to use this instead of the RNN or LSTM. Everything else (data, training loop, beam search, BLEU, checkpoints) works the same.

**How it differs.** An RNN reads a sentence one word at a time and carries a hidden state forward. A Transformer has no recurrence: every word looks at every other word directly through **self-attention**, all at once. That makes training much faster (a whole sentence in one pass instead of a loop) and makes long-distance links easier to learn. It's the architecture behind modern translation systems.

**The pieces, each its own class below:**
1. **Positional encoding.** Attention alone treats a sentence as an unordered set of words, so a fixed pattern of sines and cosines is added to each position to tell the model the word order.
2. **Multi-head attention.** The attention from section 5, run as several smaller "heads" side by side. Each head can look for a different kind of relationship (subject and verb, adjective and noun, …).
3. **Feed-forward block.** A small two-layer network applied to every position separately.
4. **Encoder layer.** Self-attention over the English words, then the feed-forward block.
5. **Decoder layer.** *Masked* self-attention over the Italian words so far (the causal mask stops it looking at future words), then attention over the English words, then the feed-forward block.

Every block is wrapped as **x + Block(Norm(x))**: normalize, apply the block, add the result back to its input. Adding the input back (a *residual connection*) lets information and gradients pass straight through deep stacks. Normalizing *before* each block ("pre-norm") trains more stably than the original paper's order.

**Decoding with beam search.** An RNN decoder carries a hidden state from word to word. A Transformer decoder has none: at each step it re-reads all the Italian words written so far. So its "state" for beam search is simply that list of words, and `step()` has the same inputs and outputs as the LSTM decoder's.

**Settings** (in `CONFIG`): `transformer_dim`, `n_heads`, `ff_dim`, `dropout`, and the number of layers from `enc_n_layers` / `dec_n_layers`. Transformers usually need 2–6 layers each and a smaller learning rate than an LSTM (e.g. 0.0005).

In [24]:
#adds a fixed pattern of sines and cosines to each position, so the model knows the word order
#input and output: (B, S, D)        D = transformer_dim (must be even)
class PositionalEncoding_Node(nn.Module):
  def __init__(self, model_dim, max_len=500):
    super(PositionalEncoding_Node, self).__init__()
    position = torch.arange(max_len).unsqueeze(1)                                              # (max_len, 1)
    frequency = torch.exp(torch.arange(0, model_dim, 2) * (-math.log(10000.0) / model_dim))    # (D/2,)
    encoding = torch.zeros(max_len, model_dim)
    encoding[:, 0::2] = torch.sin(position * frequency)     #even dimensions
    encoding[:, 1::2] = torch.cos(position * frequency)     #odd dimensions
    self.register_buffer('encoding', encoding, persistent=False)   #fixed values, not learned or saved

  def forward(self, x):
    return x + self.encoding[:x.shape[1]]


#several attention heads side by side, each comparing queries and keys in its own smaller space
#  query_input:     (B, Q, D)  the positions asking
#  key_value_input: (B, K, D)  the positions being looked at
#  key_mask:        (B, K)     True for real tokens, False for padding
#  causal:          position i may only look at positions <= i (decoder self-attention)
#returns (B, Q, D)
class MultiHeadAttention_Node(nn.Module):
  def __init__(self, model_dim, n_heads, dropout=0.1, causal=False):
    super(MultiHeadAttention_Node, self).__init__()
    if model_dim % n_heads != 0:
      raise ValueError('transformer_dim ({}) must be divisible by n_heads ({})'.format(model_dim, n_heads))
    self.n_heads = n_heads
    self.head_dim = model_dim // n_heads
    self.causal = causal

    self.W_query = nn.Linear(model_dim, model_dim)
    self.W_key = nn.Linear(model_dim, model_dim)
    self.W_value = nn.Linear(model_dim, model_dim)
    self.W_out = nn.Linear(model_dim, model_dim)     #mixes the heads' results back together
    self.dropout = nn.Dropout(dropout)

  #(B, L, D) -> (B, heads, L, head_dim): cut each vector into one slice per head
  def split_heads(self, x):
    batch_size, length, _ = x.shape
    return x.view(batch_size, length, self.n_heads, self.head_dim).transpose(1, 2)

  def forward(self, query_input, key_value_input, key_mask=None):
    query = self.split_heads(self.W_query(query_input))         # (B, h, Q, d)
    key = self.split_heads(self.W_key(key_value_input))         # (B, h, K, d)
    value = self.split_heads(self.W_value(key_value_input))     # (B, h, K, d)

    #scaled dot-product scores, one per (head, query, key) -> (B, h, Q, K)
    scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)

    if key_mask is not None:            #never attend to padding
      scores = scores.masked_fill(~key_mask[:, None, None, :], float('-inf'))
    if self.causal:                     #never attend to later positions
      n_queries, n_keys = scores.shape[-2], scores.shape[-1]
      future = torch.triu(torch.ones(n_queries, n_keys, dtype=torch.bool, device=scores.device), diagonal=1)
      scores = scores.masked_fill(future, float('-inf'))

    weights = self.dropout(F.softmax(scores, dim=-1))
    context = weights @ value                                    # (B, h, Q, d)

    #put the heads side by side again: (B, h, Q, d) -> (B, Q, h*d) = (B, Q, D)
    batch_size, _, n_queries, _ = context.shape
    context = context.transpose(1, 2).reshape(batch_size, n_queries, -1)
    return self.W_out(context)


#the same small network applied to every position separately: Linear -> ReLU -> Linear
class FeedForward_Node(nn.Module):
  def __init__(self, model_dim, ff_dim, dropout=0.1):
    super(FeedForward_Node, self).__init__()
    self.linear1 = nn.Linear(model_dim, ff_dim)
    self.linear2 = nn.Linear(ff_dim, model_dim)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [25]:
#one encoder layer (pre-norm):
#  x = x + SelfAttention(Norm(x))      every English word looks at every English word
#  x = x + FeedForward(Norm(x))
class TransformerEncoder_Node(nn.Module):
  def __init__(self, model_dim, n_heads, ff_dim, dropout=0.1):
    super(TransformerEncoder_Node, self).__init__()
    self.self_attention = MultiHeadAttention_Node(model_dim, n_heads, dropout)
    self.feed_forward = FeedForward_Node(model_dim, ff_dim, dropout)
    self.norm1 = nn.LayerNorm(model_dim)
    self.norm2 = nn.LayerNorm(model_dim)
    self.dropout = nn.Dropout(dropout)

  #x: (B, S, D), src_mask: (B, S)  ->  (B, S, D)
  def forward(self, x, src_mask):
    normed = self.norm1(x)
    x = x + self.dropout(self.self_attention(normed, normed, src_mask))
    x = x + self.dropout(self.feed_forward(self.norm2(x)))
    return x


#one decoder layer (pre-norm):
#  x = x + MaskedSelfAttention(Norm(x))                  each Italian word looks at itself and earlier Italian words
#  x = x + EncoderDecoderAttention(Norm(x), English)     each Italian word looks at the English words
#  x = x + FeedForward(Norm(x))
class TransformerDecoder_Node(nn.Module):
  def __init__(self, model_dim, n_heads, ff_dim, dropout=0.1):
    super(TransformerDecoder_Node, self).__init__()
    self.self_attention = MultiHeadAttention_Node(model_dim, n_heads, dropout, causal=True)
    self.encoder_attention = MultiHeadAttention_Node(model_dim, n_heads, dropout)
    self.feed_forward = FeedForward_Node(model_dim, ff_dim, dropout)
    self.norm1 = nn.LayerNorm(model_dim)
    self.norm2 = nn.LayerNorm(model_dim)
    self.norm3 = nn.LayerNorm(model_dim)
    self.dropout = nn.Dropout(dropout)

  #x: (B, T, D) Italian so far, encoder_outputs: (B, S, D), src_mask: (B, S), tgt_mask: (B, T)  ->  (B, T, D)
  def forward(self, x, encoder_outputs, src_mask, tgt_mask):
    normed = self.norm1(x)
    x = x + self.dropout(self.self_attention(normed, normed, tgt_mask))
    x = x + self.dropout(self.encoder_attention(self.norm2(x), encoder_outputs, src_mask))
    x = x + self.dropout(self.feed_forward(self.norm3(x)))
    return x

In [26]:
#embedding + positions + a stack of encoder layers
#inputs: (B, S) English token indices  ->  (B, S, D) one vector per English token
class TransformerEncoderNN(nn.Module):
  def __init__(self, n_layers, model_dim, n_heads, ff_dim, dropout, vocab_size):
    super(TransformerEncoderNN, self).__init__()
    self.model_dim = model_dim
    self.embedding_node = Embedding_Node(vocab_size, model_dim)
    self.positional_encoding = PositionalEncoding_Node(model_dim)
    self.dropout = nn.Dropout(dropout)
    self.layers = nn.ModuleList([TransformerEncoder_Node(model_dim, n_heads, ff_dim, dropout) for _ in range(n_layers)])
    self.final_norm = nn.LayerNorm(model_dim)     #a pre-norm stack ends with one more normalization

  def forward(self, inputs):
    src_mask = inputs != PAD_IDX
    x = self.embedding_node(inputs) * math.sqrt(self.model_dim)    #scale embeddings up to the size of the position signal
    x = self.dropout(self.positional_encoding(x))
    for layer in self.layers:
      x = layer(x, src_mask)
    return self.final_norm(x)


#embedding + positions + a stack of decoder layers + output layer
class TransformerDecoderNN(nn.Module):
  def __init__(self, n_layers, model_dim, n_heads, ff_dim, dropout, vocab_size):
    super(TransformerDecoderNN, self).__init__()
    self.n_layers = n_layers
    self.model_dim = model_dim
    self.vocab_size = vocab_size
    self.embedding_node = Embedding_Node(vocab_size, model_dim)
    self.positional_encoding = PositionalEncoding_Node(model_dim)
    self.dropout = nn.Dropout(dropout)
    self.layers = nn.ModuleList([TransformerDecoder_Node(model_dim, n_heads, ff_dim, dropout) for _ in range(n_layers)])
    self.final_norm = nn.LayerNorm(model_dim)
    self.output_layer = nn.Linear(model_dim, vocab_size)

  #every position at once (training): tgt_input (B, T) Italian tokens so far  ->  logits (B, T, V)
  #the causal mask makes position t see only tokens 0..t, so each position predicts the token after it
  def forward(self, tgt_input, encoder_outputs, src_mask):
    tgt_mask = tgt_input != PAD_IDX
    x = self.embedding_node(tgt_input) * math.sqrt(self.model_dim)
    x = self.dropout(self.positional_encoding(x))
    for layer in self.layers:
      x = layer(x, encoder_outputs, src_mask, tgt_mask)
    return self.output_layer(self.final_norm(x))

  #one decoding step, with the same inputs and outputs as DecoderNN.step, so beam search works unchanged.
  #A Transformer carries no hidden state: its "state" is the Italian tokens written so far.
  #  hidden = [tokens so far (B, t)],  cell = [None]
  def step(self, prev_word, hidden, cell, encoder_outputs, src_mask):
    words_so_far = torch.cat([hidden[0], prev_word.unsqueeze(1)], dim=1)       # (B, t+1)
    logits = self.forward(words_so_far, encoder_outputs, src_mask)[:, -1]       # (B, V) only the newest position
    return logits, [words_so_far], [None], None

In [27]:
#encoder + decoder, with the same methods as Seq2Seq (encode, forward), so training and beam search work unchanged
class TransformerSeq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super(TransformerSeq2Seq, self).__init__()
    self.encoder = encoder
    self.decoder = decoder

  #src: (B, S) -> encoder_outputs (B, S, D), src_mask (B, S), decoder state [no tokens yet], [None]
  def encode(self, src):
    src_mask = src != PAD_IDX
    encoder_outputs = self.encoder(src)
    no_tokens_yet = torch.zeros(src.shape[0], 0, dtype=torch.long, device=src.device)
    return encoder_outputs, src_mask, [no_tokens_yet], [None]

  #training with teacher forcing - same shift as Seq2Seq, but all positions in one pass instead of a loop
  #src: (B, S), tgt: (B, T)  ->  logits (B, T-1, V): a prediction for tgt[:, 1:]
  def forward(self, src, tgt):
    encoder_outputs, src_mask, _, _ = self.encode(src)
    return self.decoder(tgt[:, :-1], encoder_outputs, src_mask)


def build_transformer(config, eng_vocab_size, it_vocab_size):
  dims = (config['transformer_dim'], config['n_heads'], config['ff_dim'], config['dropout'])
  encoder = TransformerEncoderNN(config['enc_n_layers'], *dims, eng_vocab_size)
  decoder = TransformerDecoderNN(config['dec_n_layers'], *dims, it_vocab_size)
  return TransformerSeq2Seq(encoder, decoder).to(device)

**Roadmap**
- [x] batches
- [x] beam search
- [x] different attention methods (dot, scaled dot, Bahdanau)
- [x] evaluate with BLEU scores
- [x] subword segmentation (BPE, stemmer + BPE, lemma + tags)
- [x] Transformer
- [x] bidirectional encoder
- [ ] dependency parsing, pretraining, and fine-tuning

**Run time notes**
- Embedding size: common values range from 50 to 300. Larger values like 512 or 1024 can be used for more complex tasks or larger datasets.